# Creating our first "AI Agent" with LangChain 1.0
* As you will see later, this is really not an AI Agent, even when LangChain calls it Agent.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

#### Update note: current OpenAI models
* The original example is very good for learning the concept, and it worked with the OpenAI models available when the course was recorded: it used `gpt-4o-mini`.
* OpenAI has since announced the retirement of `gpt-4o-mini` (shutdown on January 14, 2027) and recommends `gpt-6-luna` as its replacement. With the current models, we would do it like this: `MODEL = "gpt-6-luna"`, and then `create_agent(model=MODEL, ...)`.
* The explanations below still show `gpt-4o-mini`: the ideas are the same, only the model name changes.
* Check the current models at [developers.openai.com/api/docs/models](https://developers.openai.com/api/docs/models).

In [ ]:
from langchain.agents import create_agent

# Original version: this example used gpt-4o-mini.
# OpenAI has deprecated it (shutdown on January 14, 2027) and recommends
# gpt-6-luna as its replacement: small, fast and affordable for practice.
MODEL = "gpt-6-luna"

system_prompt = "You are an investigative journalist."

agent = create_agent(
    model=MODEL,
    system_prompt=system_prompt
)

from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Who really killed JFK?")]}
)

print(response['messages'][-1].content)

#### Let's explain the previous code in simple terms
Let's go through it line by line and explain what each part is doing in plain English.

```python
from langchain.agents import create_agent
```

* This imports a function called `create_agent` from LangChain.
* Think of `create_agent` as a “helper” that builds an AI agent for you.

```python
system_prompt = "You are an investigative journalist."
```

* This creates a variable named `system_prompt`.
* The text inside it is an instruction that sets the *role/personality* of the AI.
* Here, you’re telling the AI: “Act like an investigative journalist.”

```python
agent = create_agent(
    model="gpt-4o-mini",
    system_prompt=system_prompt
)
```

* This creates the agent and stores it in a variable called `agent`.
* `model="gpt-4o-mini"` tells LangChain which AI model to use.
* `system_prompt=system_prompt` gives the agent the instruction you wrote above.
* After this, `agent` is basically your ready-to-use AI “journalist bot”.

```python
from langchain.messages import HumanMessage
```

* This imports `HumanMessage`, which is a LangChain class representing a message written by a human user.
* LangChain uses message objects to structure conversations (user messages, AI messages, system messages, etc.).

```python
response = agent.invoke(
    {"messages": [HumanMessage(content="Who really killed JFK?")]}
)
```

* This is where you *run* the agent.
* `invoke(...)` means: “Take this input, call the model, and give me back the result.”
* You pass in a dictionary with a key `"messages"`.
* The value is a list of messages — here it’s just one `HumanMessage`.
* That message’s `content` is the user’s question: `"Who really killed JFK?"`
* The result is stored in `response`.

```python
print(response['messages'][-1].content)
```

* `response` is a structured object (often a dictionary-like structure) that includes a list of messages.
* `response['messages']` grabs the conversation messages returned by the agent.
* `[-1]` means “the last item in the list” (usually the agent’s final reply).
* `.content` gets the actual text of that last message.
* `print(...)` displays it in your terminal.


#### Big-picture summary (in one sentence)

You create an AI agent with a “journalist” role, send it a user question as a structured message, and then print the agent’s final answer.

## Is this really an AI Agent?
**This code doesn't actually show true agentic behavior** - it's really just a fancy chatbot wrapper, even though LangChain calls it an "agent."

#### What Makes an AI "Agentic"?

A true AI agent should be able to:

1. **Make decisions autonomously** - choose what to do next without constant human instruction
2. **Use tools** - pick and use different tools (search engines, calculators, databases) as needed
3. **Plan multi-step tasks** - break down complex goals into smaller steps
4. **Iterate and adapt** - try different approaches if something doesn't work

#### What This Code Actually Does

This code just:
- Receives one question
- Generates one response
- Stops

That's a **chatbot**, not really an agent. It's like asking someone a question and getting an answer - there's no autonomous decision-making happening.

#### Where True Agentic Behavior Would Appear

A real agent version would look more like:

```python
agent = create_agent(
    model="gpt-4o-mini",
    tools=[web_search, calculator, database_query],  # ← Tools it can use
    system_prompt=system_prompt
)
```

Then when you ask "Who killed JFK?", the agent might:
1. **Decide** "I need current information" 
2. **Use** the web_search tool
3. **Analyze** the results
4. **Decide** "I need historical records too"
5. **Use** the database_query tool
6. **Synthesize** everything into an answer

**The key difference:** An agent chooses its own path to solve your problem, using whatever tools it thinks are necessary. This example just answers directly without any autonomous tool use or decision-making.

## Using .stream instead of .invoke

In [ ]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Really, who really killed JFK?")]},
    stream_mode="messages"
):
    
    if token.content:
        print(token.content, end="", flush=True)

## Let's explain the previous code in simple terms
**Streaming** is where agent behavior starts to *feel alive*.

We will explain it **line by line**, then give you the **big idea** at the end.


#### Line-by-line explanation

```python
for token, metadata in agent.stream(
```

* This starts a `for` loop.
* Instead of getting **one final answer**, you are asking the agent to **stream its response piece by piece**.
* Each loop iteration gives you:

  * `token` → a small chunk of the AI’s output
  * `metadata` → extra information about that chunk (often ignored at beginner level)

---

```python
    {"messages": [HumanMessage(content="Really, who really killed JFK?")]},
```

* This is the input you send to the agent.
* Same idea as before:

  * You pass a list of messages
  * Here it’s one human message
* The question is slightly different, but the structure is identical.

---

```python
    stream_mode="messages"
```

* This tells LangChain **how** you want the output streamed.
* `"messages"` means:

  * “Stream structured message chunks, not raw text”
* Each `token` you receive will behave like a partial AI message object.

---

```python
):
```

* This closes the `agent.stream(...)` call.
* At this point, LangChain starts calling the model and yielding output chunks.

---

```python
    if token.content:
```

* Not every streamed chunk contains text.
* Some chunks may be:

  * control signals
  * empty
  * metadata-only
* This line checks:

  > “Does this chunk actually contain text?”

---

```python
        print(token.content, end="", flush=True)
```

* This prints the chunk **immediately**, without a new line.
* `end=""`:

  * Prevents line breaks so text appears continuously.
* `flush=True`:

  * Forces Python to print instantly instead of buffering.
* Result:

  * The answer appears **word by word**, like someone typing.

---

#### What’s really happening under the hood

Instead of this:

```
[ agent thinks ]
[ agent finishes ]
[ you get the full answer ]
```

You now get this:

```
[ agent starts ]
"Well,"
" based"
" on"
" available"
" evidence"
...
```

So you are **watching the agent speak in real time**.

---

#### Why this is important for agents (it is not just for a better User Experience)

Streaming is not just a “cool effect”.

It allows:

* real-time feedback
* interruption or cancellation
* chaining agents together while one is still talking
* building chat UIs, terminals, or voice systems

In agent systems, streaming is often used to:

* observe reasoning
* pipe partial output into another agent
* react dynamically to what’s being said

---

#### Beginner mental model (very useful)

Think of it like this:

* `invoke()` → **Give me the finished answer**
* `stream()` → **Let me listen while you talk**

Same agent. Same role. Different interaction mode.

---

## One-sentence summary

This code asks the agent to answer a question **incrementally**, and prints each small piece of its response as soon as it is generated, creating a live, streaming output.

## How to run this code from Visual Studio Code
* Open Terminal.
* Make sure you are in the project folder.
* Make sure you have the poetry env activated.
* Enter and run the following command:
    * `python 002-creating-an-agent.py` 